In [4]:
import numpy as np
import random


random.seed(42)
np.random.seed(42)

cities = np.array([
    [1, 1],
    [2, 5],
    [5, 2],
    [6, 6],
    [8, 3],
    [3, 7]
])

NUM_CITIES = len(cities)


distance = np.zeros((NUM_CITIES, NUM_CITIES))

for i in range(NUM_CITIES):
    for j in range(NUM_CITIES):
        distance[i][j] = np.linalg.norm(cities[i] - cities[j])


NUM_ANTS = 10
NUM_ITERATIONS = 50
ALPHA = 1.0
BETA = 2.0
RHO = 0.5
Q = 100


pheromone = np.ones((NUM_CITIES, NUM_CITIES))
np.fill_diagonal(pheromone, 0)

heuristic = 1 / (distance + 1e-10)
np.fill_diagonal(heuristic, 0)

# 4. Select next city probabilistically
def select_next_city(current_city, unvisited):
    unvisited = list(unvisited)

    weights = [
        (pheromone[current_city][city] ** ALPHA) *
        (heuristic[current_city][city] ** BETA)
        for city in unvisited
    ]

    return random.choices(unvisited, weights=weights, k=1)[0]


def construct_tour():
    start_city = random.randint(0, NUM_CITIES - 1)
    tour = [start_city]
    unvisited = set(range(NUM_CITIES))
    unvisited.remove(start_city)

    current_city = start_city

    while unvisited:
        next_city = select_next_city(current_city, unvisited)
        tour.append(next_city)
        unvisited.remove(next_city)
        current_city = next_city

    # Return to starting city
    tour.append(start_city)
    return tour


def calculate_tour_length(tour):
    return sum(
        distance[tour[i]][tour[i + 1]]
        for i in range(len(tour) - 1)
    )


def update_pheromones(all_tours):
    global pheromone


    pheromone *= (1 - RHO)

   
    for tour, tour_length in all_tours:
        deposit = Q / tour_length

        for i in range(len(tour) - 1):
            a, b = tour[i], tour[i + 1]
            pheromone[a][b] += deposit
            pheromone[b][a] += deposit

    np.fill_diagonal(pheromone, 0)

best_tour = None
best_length = float("inf")
convergence = []

for iteration in range(NUM_ITERATIONS):
    all_tours = []

    for ant in range(NUM_ANTS):
        tour = construct_tour()
        tour_length = calculate_tour_length(tour)
        all_tours.append((tour, tour_length))

   
        if tour_length < best_length:
            best_length = tour_length
            best_tour = tour.copy()


    update_pheromones(all_tours)
    convergence.append(best_length)

    if (iteration + 1) % 10 == 0:
        print(
            f"Iteration {iteration + 1}: "
            f"Best Distance = {best_length:.2f}"
        )


print("\n========== FINAL RESULT ==========")
print("Best Tour:", best_tour)
print("Shortest Distance:", round(best_length, 2))

Iteration 10: Best Distance = 20.41
Iteration 20: Best Distance = 20.41
Iteration 30: Best Distance = 20.41
Iteration 40: Best Distance = 20.41
Iteration 50: Best Distance = 20.41

========== FINAL RESULT ==========
Best Tour: [2, 0, 1, 5, 3, 4, 2]
Shortest Distance: 20.41
